In [ ]:
import os
import csv
import cv2
import numpy as np
from collections import defaultdict, deque

In [ ]:
VIDEO_NAME  = "Road traffic video for object recognition_part_1.mp4"
VIDEO_PATH  = os.path.join("Downloads", "Segments", VIDEO_NAME)
OUTPUT_PATH = "traffic_speed_output_opencv.mp4"
CSV_PATH    = "vehicle_speeds.csv"

In [ ]:
# Known frame rate of the source video.
FPS: float = 25.0

# Minimum contour area (px²) to be counted as a vehicle.
MIN_CONTOUR_AREA = 1_500

# How many frames of BEV position to keep for speed smoothing.
# At 25 fps: 8 frames = 0.32 s  (good for a 10 s clip).
SPEED_WINDOW: int = 8

# Minimum frames a track must have before we report a speed.
MIN_FRAMES_FOR_SPEED: int = 5

# ---------------------------------------------------------------------------
# GREEN-LINE TRIPWIRE POSITIONS  (pixel coords in the 1280×720 frame)
# ---------------------------------------------------------------------------
# *** Adjust these to match the EXACT green lines visible in YOUR video ***
# Each green line is 25 m in real-world length (given by the assignment).

GREEN_LINE_LENGTH_M = 25.0

# Upper green line (Line 1 – further from camera)
LINE_1_LEFT  = np.array([340, 400], dtype=np.float32)
LINE_1_RIGHT = np.array([940, 400], dtype=np.float32)

# Lower green line (Line 2 – closer to camera)
LINE_2_LEFT  = np.array([100, 560], dtype=np.float32)
LINE_2_RIGHT = np.array([1180, 560], dtype=np.float32)

LINE_1_Y = int(LINE_1_LEFT[1])
LINE_2_Y = int(LINE_2_LEFT[1])

# ---------------------------------------------------------------------------
# LANE-DASH CALIBRATION  (used by rolling-window BEV method)
# ---------------------------------------------------------------------------
# Ground-truth: every 3 consecutive dashes span 17 m.
# Motorway visible from bridge: ~6 dash-groups top-to-bottom → ~102 m.
DASH_GROUP_M: float        = 17.0
DASH_GROUPS_IN_FRAME: int  = 6       # count from the video (top edge → bottom edge)
_ROAD_LENGTH_M: float      = DASH_GROUPS_IN_FRAME * DASH_GROUP_M   # 102 m

# Road width for the full dual carriageway captured in frame
_ROAD_WIDTH_M: float = 25.0   # ~4 lanes × 3.65 m + median + shoulders ≈ 28 m

# ---------------------------------------------------------------------------
# PERSPECTIVE CALIBRATION  (re-calibrated to 1280×720)
# ---------------------------------------------------------------------------
# Corners of the road rectangle visible in the 1280×720 frame, ordered:
#   0 = top-left  (far, left shoulder)
#   1 = top-right (far, right shoulder)
#   2 = bottom-right (near, right shoulder)
#   3 = bottom-left  (near, left shoulder)
#
# Values read from diag_frame_060.jpg pixel grid:
SOURCE = np.array([
    [ 340, 400],   # top-left  (far left edge of road at horizon)
    [ 940, 400],   # top-right (far right edge of road at horizon)
    [100, 560],   # bottom-right (near right shoulder)
    [  1180, 560],   # bottom-left  (near left shoulder)
], dtype=np.float32)

# Corresponding real-world rectangle (metres).
TARGET = np.array([
    [           0,             0],
    [_ROAD_WIDTH_M,             0],
    [_ROAD_WIDTH_M, _ROAD_LENGTH_M],
    [           0, _ROAD_LENGTH_M],
], dtype=np.float32)

PERSPECTIVE_MATRIX = cv2.getPerspectiveTransform(SOURCE, TARGET)

# ---------------------------------------------------------------------------
# PERSPECTIVE-CORRECTED TRIPWIRE DISTANCE  (derived from the green lines)
# ---------------------------------------------------------------------------
#
# Camera model (flat road, pinhole camera at height H):
#     Road width at pixel row y :  L(y) = W·(y − y_v) / H
#     Distance along road       :  Z(y) = H·f / (y − y_v)
#
# Given L1, L2 (pixel widths of the two 25 m green lines) and y1, y2:
#     y_v = (L1·y₂ − L2·y₁) / (L1 − L2)          (vanishing point Y)
#     D   = W·(y₂ − y₁) / [ L1·(y₂ − y_v) ]      (metres, corrected)
# ---------------------------------------------------------------------------

_L1 = float(np.linalg.norm(LINE_1_RIGHT - LINE_1_LEFT))
_L2 = float(np.linalg.norm(LINE_2_RIGHT - LINE_2_LEFT))
_y1 = float(LINE_1_LEFT[1])
_y2 = float(LINE_2_LEFT[1])

_y_v = (_L1 * _y2 - _L2 * _y1) / (_L1 - _L2)

TRIPWIRE_DISTANCE_M = GREEN_LINE_LENGTH_M * (_y2 - _y1) / (_L1 * (_y2 - _y_v))
NAIVE_DISTANCE_M    = GREEN_LINE_LENGTH_M * (_y2 - _y1) / ((_L1 + _L2) / 2.0)


In [ ]:
# ---------------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------------

def transform_point(pt: np.ndarray) -> np.ndarray:
    """Map a single (x, y) pixel to the bird's-eye-view plane (metres)."""
    src = np.array([[[float(pt[0]), float(pt[1])]]], dtype=np.float32)
    dst = cv2.perspectiveTransform(src, PERSPECTIVE_MATRIX)
    return dst[0, 0]   # shape (2,)  → (x_m, y_m)


class CentroidTracker:
    """
    Lightweight greedy centroid tracker.
    Assigns consistent IDs across frames using nearest-neighbour matching.
    """

    def __init__(self, max_lost: int = 8, max_distance: int = 120):
        self.next_id      = 0
        self.centroids    = {}               # id → (cx, cy)
        self.lost_count   = defaultdict(int)
        self.max_lost     = max_lost
        self.max_distance = max_distance

    def update(self, rects: list[tuple[int, int, int, int]]) -> dict:
        """
        Parameters
        ----------
        rects : list of (x, y, w, h) bounding boxes.

        Returns
        -------
        dict  track_id → (cx, cy)  for every active track.
        """
        if not rects:
            for tid in list(self.centroids):
                self.lost_count[tid] += 1
                if self.lost_count[tid] > self.max_lost:
                    del self.centroids[tid]
                    del self.lost_count[tid]
            return dict(self.centroids)

        new_cents = [(int(x + w / 2), int(y + h / 2)) for (x, y, w, h) in rects]

        if not self.centroids:
            for nc in new_cents:
                self.centroids[self.next_id] = nc
                self.lost_count[self.next_id] = 0
                self.next_id += 1
            return dict(self.centroids)

        ex_ids  = list(self.centroids)
        ex_pts  = [self.centroids[i] for i in ex_ids]
        used_ex = set()
        used_nw = set()

        # Build distance matrix and sort
        dist_mat = np.linalg.norm(
            np.array(ex_pts)[:, None, :] - np.array(new_cents)[None, :, :],
            axis=-1,
        )   # shape (n_existing, n_new)

        pairs = sorted(
            [(dist_mat[i, j], i, j)
             for i in range(len(ex_pts))
             for j in range(len(new_cents))],
            key=lambda t: t[0],
        )

        for d, ei, ni in pairs:
            if ei in used_ex or ni in used_nw:
                continue
            if d > self.max_distance:
                break
            tid = ex_ids[ei]
            self.centroids[tid]  = new_cents[ni]
            self.lost_count[tid] = 0
            used_ex.add(ei)
            used_nw.add(ni)

        for ei, tid in enumerate(ex_ids):
            if ei not in used_ex:
                self.lost_count[tid] += 1
                if self.lost_count[tid] > self.max_lost:
                    del self.centroids[tid]
                    del self.lost_count[tid]

        for ni, nc in enumerate(new_cents):
            if ni not in used_nw:
                self.centroids[self.next_id] = nc
                self.lost_count[self.next_id] = 0
                self.next_id += 1

        return dict(self.centroids)


# ---------------------------------------------------------------------------
# SPEED STATS  (per track, accumulated across whole video)
# ---------------------------------------------------------------------------

class SpeedStats:
    """Accumulates per-frame speed samples and computes avg / min / max."""

    def __init__(self):
        self._samples: list[float] = []

    def add(self, kmh: float) -> None:
        self._samples.append(kmh)

    @property
    def avg(self) -> float | None:
        return float(np.mean(self._samples)) if self._samples else None

    @property
    def min(self) -> float | None:
        return float(np.min(self._samples)) if self._samples else None

    @property
    def max(self) -> float | None:
        return float(np.max(self._samples)) if self._samples else None

    @property
    def count(self) -> int:
        return len(self._samples)


# ---------------------------------------------------------------------------
# TRIPWIRE CROSSING LOGIC
# ---------------------------------------------------------------------------

class TripwireCrossing:
    """
    Detects when a tracked centroid crosses each horizontal tripwire line
    and computes the speed from the crossing-time difference.

    Reports BOTH perspective-corrected and naive speeds.
    """

    def __init__(self, line_1_y: int, line_2_y: int,
                 corrected_dist_m: float, naive_dist_m: float, fps: float):
        self.line_1_y         = line_1_y
        self.line_2_y         = line_2_y
        self.corrected_dist_m = corrected_dist_m
        self.naive_dist_m     = naive_dist_m
        self.fps              = fps

        # Per-track state
        self.prev_y:       dict[int, int]   = {}
        self.crossed_1:    dict[int, int]   = {}   # tid → frame when crossed line 1
        self.speed_corr:   dict[int, float] = {}   # tid → corrected speed (km/h)
        self.speed_naive:  dict[int, float] = {}   # tid → naive speed (km/h)

    def update(self, active: dict[int, tuple[int, int]], frame_idx: int):
        """Call once per frame with the active centroid dict."""
        for tid, (cx, cy) in active.items():
            prev = self.prev_y.get(tid)
            self.prev_y[tid] = cy

            if prev is None:
                continue

            # Detect crossing of Line 1 (in either direction)
            if tid not in self.crossed_1:
                if prev <= self.line_1_y < cy or cy <= self.line_1_y < prev:
                    self.crossed_1[tid] = frame_idx

            # Detect crossing of Line 2 (only after Line 1 was already crossed)
            elif tid not in self.speed_corr:
                if prev <= self.line_2_y < cy or cy <= self.line_2_y < prev:
                    elapsed_frames = frame_idx - self.crossed_1[tid]
                    if elapsed_frames > 0:
                        elapsed_s = elapsed_frames / self.fps
                        self.speed_corr[tid]  = (self.corrected_dist_m / elapsed_s) * 3.6
                        self.speed_naive[tid] = (self.naive_dist_m     / elapsed_s) * 3.6

    def get_speed(self, tid: int) -> float | None:
        """Return the perspective-corrected tripwire speed, or None."""
        return self.speed_corr.get(tid)


# ---------------------------------------------------------------------------
# DRAWING
# ---------------------------------------------------------------------------
PALETTE = [
    (255,  85,  85),   # coral-red
    ( 85, 255,  85),   # lime-green
    ( 85,  85, 255),   # blue
    (255, 200,   0),   # golden
    (  0, 220, 220),   # cyan
    (220,   0, 220),   # magenta
    (255, 160,  10),   # orange
    (160, 255,  10),   # yellow-green
]

def id_color(tid: int) -> tuple[int, int, int]:
    return PALETTE[tid % len(PALETTE)]


def draw_detection(frame, rect, tid, speed_bev, speed_trip):
    """Draw bounding box with BEV speed and tripwire speed labels."""
    x, y, w, h = rect
    color = id_color(tid)
    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

    # Build label: show tripwire speed if available, else BEV speed, else "--"
    label = f"#{tid}"
    if speed_trip is not None:
        label += f"  {int(speed_trip)} km/h"
    elif speed_bev is not None:
        label += f"  ~{int(speed_bev)} km/h"
    else:
        label += "  --"

    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    bg_y1 = max(y - th - 8, 0)
    cv2.rectangle(frame, (x, bg_y1), (x + tw + 6, y), color, -1)
    cv2.putText(frame, label, (x + 3, max(y - 4, th)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1, cv2.LINE_AA)


def draw_tripwires(frame):
    """Draw the two green tripwire lines on the frame."""
    green = (0, 255, 0)
    cv2.line(frame, tuple(LINE_1_LEFT.astype(int)), tuple(LINE_1_RIGHT.astype(int)), green, 2)
    cv2.line(frame, tuple(LINE_2_LEFT.astype(int)), tuple(LINE_2_RIGHT.astype(int)), green, 2)
    cv2.putText(frame, "Line 1 (25 m)", (int(LINE_1_RIGHT[0]) + 10, int(LINE_1_RIGHT[1]) + 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, green, 1, cv2.LINE_AA)
    cv2.putText(frame, "Line 2 (25 m)", (int(LINE_2_RIGHT[0]) + 10, int(LINE_2_RIGHT[1]) + 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, green, 1, cv2.LINE_AA)


def draw_hud(frame, vehicle_count: int, frame_idx: int, total: int):
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (440, 100), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.putText(frame, f"Vehicles in frame: {vehicle_count}",
                (20, 37), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (220, 220, 220), 2, cv2.LINE_AA)
    pct = frame_idx / total * 100 if total else 0
    cv2.putText(frame, f"Frame {frame_idx}/{total}  ({pct:.0f}%)",
                (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (160, 160, 160), 1, cv2.LINE_AA)
    cv2.putText(frame, f"Tripwire dist: {TRIPWIRE_DISTANCE_M:.1f}m (corrected)"
                       f"  |  {NAIVE_DISTANCE_M:.1f}m (naive)",
                (20, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (100, 220, 100), 1, cv2.LINE_AA)



In [ ]:
def main():
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {VIDEO_PATH}")

    fps    = FPS
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video  : {VIDEO_PATH}")
    print(f"Size   : {width}x{height}  |  FPS: {fps}  |  Frames: {total}  ({total/fps:.1f}s)")
    print(f"Output : {OUTPUT_PATH}  |  CSV: {CSV_PATH}")

    # -- Background subtractor ------------------------------------------------
    bg_sub = cv2.createBackgroundSubtractorMOG2(
        history=150, varThreshold=40, detectShadows=True
    )

    # Morphological kernels (tuned for 1280×720)
    k_open   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    k_close  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    k_dilate = cv2.getStructuringElement(cv2.MORPH_RECT,    (15, 15))

    # -- Tracker & histories --------------------------------------------------
    tracker = CentroidTracker(max_lost=8, max_distance=100)

    # Tripwire crossing detector
    tripwire = TripwireCrossing(
        LINE_1_Y, LINE_2_Y,
        TRIPWIRE_DISTANCE_M, NAIVE_DISTANCE_M,
        fps,
    )

    # BEV Y-coordinate history (for rolling-window speed as fallback)
    bev_history:   dict[int, deque]     = defaultdict(lambda: deque(maxlen=SPEED_WINDOW))
    # Accumulated BEV speed stats per track
    speed_stats:   dict[int, SpeedStats] = defaultdict(SpeedStats)
    # Last known rect per track
    tid_to_rect:   dict[int, tuple]     = {}
    # Frame-level instant BEV speed per track (for display before tripwire fires)
    instant_speed: dict[int, float]     = {}

    # -- Video writer ---------------------------------------------------------
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # --- 1. Foreground mask ----------------------------------------------
        fg = bg_sub.apply(frame)
        _, fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)   # remove shadows
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN,  k_open)
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, k_close)
        fg = cv2.dilate(fg, k_dilate, iterations=1)

        # --- 2. Contour → bounding boxes -------------------------------------
        contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        rects = [cv2.boundingRect(c) for c in contours
                 if cv2.contourArea(c) >= MIN_CONTOUR_AREA]

        # --- 3. Track --------------------------------------------------------
        active = tracker.update(rects)

        # Rebuild tid → rect
        new_tid_to_rect: dict[int, tuple] = {}
        used_ri = set()
        for tid, (cx, cy) in active.items():
            best_d, best_i = float("inf"), None
            for ri, (rx, ry, rw, rh) in enumerate(rects):
                if ri in used_ri:
                    continue
                d = ((cx - (rx + rw//2))**2 + (cy - (ry + rh//2))**2) ** 0.5
                if d < best_d:
                    best_d, best_i = d, ri
            if best_i is not None and best_d < 100:
                new_tid_to_rect[tid] = rects[best_i]
                used_ri.add(best_i)
            elif tid in tid_to_rect:
                new_tid_to_rect[tid] = tid_to_rect[tid]
        tid_to_rect = new_tid_to_rect

        # --- 4a. Rolling-window BEV speed (fallback / real-time display) -----
        instant_speed.clear()

        for tid, (cx, cy) in active.items():
            bev = transform_point(np.array([cx, cy], dtype=np.float32))
            bev_history[tid].append(float(bev[1]))   # Y = along-road distance (m)

            h = list(bev_history[tid])
            if len(h) >= MIN_FRAMES_FOR_SPEED:
                diffs = [abs(h[i + 1] - h[i]) for i in range(len(h) - 1)]
                median_disp_m = float(np.median(diffs))
                spd = median_disp_m * fps * 3.6
                if 1.0 < spd < 180.0:
                    instant_speed[tid] = spd
                    speed_stats[tid].add(spd)

        # --- 4b. Tripwire crossing detection ---------------------------------
        tripwire.update(active, frame_idx)

        # --- 5. Annotate & write frame ----------------------------------------
        annotated = frame.copy()

        # Draw green tripwire lines
        draw_tripwires(annotated)

        for tid, (cx, cy) in active.items():
            rect = tid_to_rect.get(tid)
            if rect is None:
                continue
            draw_detection(annotated, rect, tid,
                           instant_speed.get(tid),
                           tripwire.get_speed(tid))

        # Count only vehicles whose centroid is inside the tripwire zone
        in_zone = sum(1 for _, (_, cy) in active.items()
                      if LINE_1_Y <= cy <= LINE_2_Y)
        draw_hud(annotated, in_zone, frame_idx, total)
        writer.write(annotated)

        if frame_idx % 50 == 0 or frame_idx == total:
            pct = frame_idx / total * 100 if total else 0
            print(f"  Frame {frame_idx}/{total}  ({pct:.0f}%)", end="\r")

    cap.release()
    writer.release()

    # --- 6. Write CSV --------------------------------------------------------
    # Collect ALL vehicle IDs from both methods
    all_tids = sorted(set(
        [tid for tid, s in speed_stats.items() if s.count > 0]
        + list(tripwire.speed_corr.keys())
    ))

    with open(CSV_PATH, "w", newline="") as f:
        writer_csv = csv.writer(f)
        writer_csv.writerow([
            "vehicle_id",
            "tripwire_speed_corrected_kmh",
            "tripwire_speed_naive_kmh",
            "bev_avg_speed_kmh",
            "bev_min_speed_kmh",
            "bev_max_speed_kmh",
        ])
        for tid in all_tids:
            s = speed_stats.get(tid)
            writer_csv.writerow([
                tid,
                f"{tripwire.speed_corr.get(tid, '')}"
                    if tid in tripwire.speed_corr else "",
                f"{tripwire.speed_naive.get(tid, '')}"
                    if tid in tripwire.speed_naive else "",
                f"{s.avg:.1f}" if s and s.count else "",
                f"{s.min:.1f}" if s and s.count else "",
                f"{s.max:.1f}" if s and s.count else "",
            ])

    n_trip = len(tripwire.speed_corr)
    n_bev  = sum(1 for s in speed_stats.values() if s.count > 0)

    print(f"\nDone!")
    print(f"  Video : {OUTPUT_PATH}")
    print(f"  CSV   : {CSV_PATH}  ({n_trip} tripwire + {n_bev} BEV measurements)")

    if n_trip > 0:
        avg_corr  = np.mean(list(tripwire.speed_corr.values()))
        avg_naive = np.mean(list(tripwire.speed_naive.values()))
        print(f"\n  Tripwire avg (perspective-corrected): {avg_corr:.1f} km/h")
        print(f"  Tripwire avg (naive / no correction): {avg_naive:.1f} km/h")
        print(f"  Perspective correction effect:         {((avg_corr/avg_naive)-1)*100:+.1f} %")